# Werewolf Agent quickstart

## Goal

決定的なDomain APIとFakeListChatModelを使い、外部APIなしで人狼ゲームを1局実行します。公開状態と本人のobservationを区別し、失敗した操作の原子性、snapshotの復元、seedによる再現性も確認します。

## Setup

このNotebookはリポジトリの依存環境で上から順に実行します。Docker、Supabase、環境変数、有料LLMは使用しません。

In [1]:
from dataclasses import asdict

from werewolf_agent import Action, Game, RuleViolation
from werewolf_demo import FakeGameDemo

SEED = 7
TEMPLATE_ID = "standard_6"

## Steps

### 1. 設定済みゲームを作成する

完全状態は`Game`が所有します。ここでは役職を表示せず、進行に必要な公開可能項目だけを確認します。

In [2]:
demo = FakeGameDemo.create(template_id=TEMPLATE_ID, seed=SEED)
snapshot = demo.game.snapshot()
observations = [demo.game.view_for(player_id) for player_id in snapshot.players]
observation = next(item for item in observations if item.available_actions)

{
    "phase": snapshot.phase.value,
    "day": snapshot.day,
    "player_count": len(snapshot.players),
    "observation_player": observation.me.name,
    "available_actions": [action.key for action in observation.available_actions],
    "legal_target_counts": {key: len(value) for key, value in observation.legal_targets.items()},
}

{'phase': 'night',
 'day': 1,
 'player_count': 6,
 'observation_player': '結衣',
 'available_actions': ['use_ability:night_attack', 'pass'],
 'legal_target_counts': {'use_ability:night_attack': 5}}

### 2. Fake agentで1操作進める

Fake modelも通常のprompt生成、schema検証、合法手検証を通ります。未解決の夜行動や投票はactorと対象を表示しません。

In [3]:
first_step = demo.step()
{
    "operation": first_step.operation,
    "phase": first_step.phase,
    "action": first_step.action_key,
    "private_action_omitted": first_step.private_action_omitted,
    "validation_status": first_step.decision.validation_status,
    "fallback_used": first_step.decision.fallback_used,
}

{'operation': 'action',
 'phase': 'night',
 'action': 'private_action',
 'private_action_omitted': True,
 'validation_status': 'valid',
 'fallback_used': False}

### 3. 一局を完走する

保存する出力は勝者、操作数、公開event数などの短い要約に限定します。

In [4]:
result = demo.run()
{
    "completed": result.completed,
    "stop_reason": result.stop_reason,
    "winner": result.winner_id,
    "finished_day": result.day,
    "actions": result.action_count,
    "phases": result.phase_count,
    "public_events": len(result.public_events),
    "decisions": len(result.decisions),
}

{'completed': True,
 'stop_reason': 'finished',
 'winner': 'village',
 'finished_day': 5,
 'actions': 46,
 'phases': 12,
 'public_events': 40,
 'decisions': 46}

## Checks

### 4. 失敗した操作とsnapshot復元を確認する

In [5]:
check_demo = FakeGameDemo.create(template_id=TEMPLATE_ID, seed=SEED)
before_invalid_action = check_demo.game.snapshot()
try:
    check_demo.game.submit(Action.vote("unknown-player", "unknown-target"))
except RuleViolation as error:
    violation_code = error.code

after_invalid_action = check_demo.game.snapshot()
restored = Game.restore(before_invalid_action, rules=check_demo.rules)
{
    "violation_code": violation_code,
    "failed_action_is_atomic": before_invalid_action == after_invalid_action,
    "snapshot_is_immutable": type(before_invalid_action.players).__name__,
    "restore_matches_snapshot": restored.snapshot() == before_invalid_action,
}

{'violation_code': 'action_not_available',
 'failed_action_is_atomic': True,
 'snapshot_is_immutable': 'FrozenDict',
 'restore_matches_snapshot': True}

### 5. 再現性と安全なresultを確認する

In [6]:
repeated_result = FakeGameDemo.create(template_id=TEMPLATE_ID, seed=SEED).run()
decision_fields = set(asdict(result.decisions[0]))
private_trace_fields = {
    "prompt_messages",
    "request_payload",
    "raw_response",
    "parsed_decision",
    "target_id",
    "role_id",
}
{
    "same_seed_same_checksum": result.checksum == repeated_result.checksum,
    "private_trace_fields_absent": not (decision_fields & private_trace_fields),
    "decision_fields": sorted(decision_fields),
}

{'same_seed_same_checksum': True,
 'private_trace_fields_absent': True,
 'decision_fields': ['day',
  'fallback_used',
  'model',
  'phase',
  'provider',
  'provider_error',
  'validation_status']}

## Next Steps

`SEED`や`TEMPLATE_ID`を変更すると別の決定的な一局を確認できます。製品コードでは、完全状態の変更を`Game.submit()`と`Game.advance()`に限定し、利用者へ返す情報は公開状態、public timeline、認証した本人のobservationに分けてください。